# z616 - DTW + Clustering por LOTES + LightGBM por cluster (cliente-producto)
~735k series -- DTW pareado completo es inviable (2.7x10^11 pares). Se subdivide en lotes de tamano manejable, DTW dentro de cada lote, clustering jerarquico dentro de cada lote.

**ADVERTENCIA DE TIEMPO:** con ~735k series y lotes de 2000, son ~368 lotes. Cada lote hace una matriz DTW de ~2M pares. Puede tardar HORAS en total. Corre `TEST_RAPIDO=True` primero para medir tiempo por lote antes de lanzar todo.

In [1]:
!pip install -q dtaidistance scipy scikit-learn lightgbm pyarrow polars

In [2]:
import os
import time
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings("ignore")

In [16]:
PARAM = {
    'experimento': 'LGB11_DTW_CP' + ('_TEST' if PARAM['TEST_RAPIDO'] else ''),
    'kaggle_competition': 'labo-iii-2026-ba',
    'features_path': '/home/ds/exp/CP602/tb_features_CP602.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'semilla': 102103,
    'batch_size': 2000,
    'k_subclusters_por_lote': 5,
    'min_filas_por_cluster': 100,
    'TEST_RAPIDO': False   # si True, procesa solo los primeros 3 lotes para medir tiempo antes de correr todo
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB11_DTW_CP_TEST


## 1. Armar la matriz serie x periodo (por par cliente-producto)

In [17]:
df = pl.read_parquet(PARAM['features_path'])

claves = df.select(["customer_id", "product_id"]).unique().sort(["customer_id", "product_id"])
claves = claves.with_row_index("fila_id")
print("series totales:", claves.height)

df = df.join(claves, on=["customer_id", "product_id"], how="left")

series totales: 698705


In [18]:
wide = df.select(["fila_id", "periodo", "tn"]).pivot(
    values="tn", index="fila_id", columns="periodo"
).sort("fila_id")

fila_ids = wide["fila_id"].to_numpy()
matriz = wide.drop("fila_id").to_numpy().astype(np.float64)
matriz = np.nan_to_num(matriz, nan=0.0)
print(matriz.shape)

(698705, 37)


## 2. Lotes: agrupar en batches consecutivos por fila_id
Division simple en bloques de tamano fijo. No es la unica forma de armar lotes, pero es la mas barata (no requiere un paso previo de pre-agrupamiento).

In [19]:
BATCH = PARAM['batch_size']
n_series = matriz.shape[0]
n_lotes = int(np.ceil(n_series / BATCH))
print("lotes totales:", n_lotes)

lotes_a_procesar = range(3) if PARAM['TEST_RAPIDO'] else range(n_lotes)

lotes totales: 350


## 3. DTW + clustering DENTRO de cada lote
Cluster final = `lote_id * k_subclusters + subcluster_local`. Mide tiempo por lote para poder estimar el total antes de lanzar todo.

In [20]:
K = PARAM['k_subclusters_por_lote']
asignaciones = np.full(n_series, -1, dtype=np.int64)

for lote_id in lotes_a_procesar:
    t0 = time.time()
    ini = lote_id * BATCH
    fin = min(ini + BATCH, n_series)

    bloque = matriz[ini:fin]
    medias = bloque.mean(axis=1, keepdims=True)
    stds = bloque.std(axis=1, keepdims=True)
    stds[stds == 0] = 1.0
    bloque_escalado = (bloque - medias) / stds

    series_lista = [bloque_escalado[i] for i in range(bloque_escalado.shape[0])]
    dist_matrix = dtw.distance_matrix_fast(series_lista)

    # espejar y sanear infinitos (misma correccion que z615)
    dist_matrix = np.where(np.isfinite(dist_matrix), dist_matrix, dist_matrix.T)
    np.fill_diagonal(dist_matrix, 0.0)
    finitos = dist_matrix[np.isfinite(dist_matrix)]
    max_finita = np.max(finitos) if finitos.size > 0 else 1.0
    dist_matrix = np.where(np.isfinite(dist_matrix), dist_matrix, max_finita)
    np.fill_diagonal(dist_matrix, 0.0)

    dist_condensada = squareform(dist_matrix, checks=False)

    Z = linkage(dist_condensada, method="average")
    k_efectivo = min(K, fin - ini)
    sub_clusters = fcluster(Z, t=k_efectivo, criterion="maxclust")

    asignaciones[ini:fin] = lote_id * K + sub_clusters

    print(f"lote {lote_id} ({fin-ini} series): {time.time()-t0:.1f}s")

if PARAM['TEST_RAPIDO']:
    print("\nTEST_RAPIDO activo -- solo se procesaron 3 lotes. Multiplicar el tiempo promedio por", n_lotes, "para estimar el total, y poner TEST_RAPIDO=False para correr todo.")

lote 0 (2000 series): 3.3s
lote 1 (2000 series): 3.2s
lote 2 (2000 series): 3.2s
lote 3 (2000 series): 3.2s
lote 4 (2000 series): 3.2s
lote 5 (2000 series): 3.2s
lote 6 (2000 series): 3.2s
lote 7 (2000 series): 3.2s
lote 8 (2000 series): 3.2s
lote 9 (2000 series): 3.2s
lote 10 (2000 series): 3.2s
lote 11 (2000 series): 3.2s
lote 12 (2000 series): 3.2s
lote 13 (2000 series): 3.2s
lote 14 (2000 series): 3.2s
lote 15 (2000 series): 3.2s
lote 16 (2000 series): 3.2s
lote 17 (2000 series): 3.2s
lote 18 (2000 series): 3.2s
lote 19 (2000 series): 3.2s
lote 20 (2000 series): 3.2s
lote 21 (2000 series): 3.2s
lote 22 (2000 series): 3.2s
lote 23 (2000 series): 3.2s
lote 24 (2000 series): 3.2s
lote 25 (2000 series): 3.2s
lote 26 (2000 series): 3.2s
lote 27 (2000 series): 3.2s
lote 28 (2000 series): 3.3s
lote 29 (2000 series): 3.2s
lote 30 (2000 series): 3.3s
lote 31 (2000 series): 3.2s
lote 32 (2000 series): 3.2s
lote 33 (2000 series): 3.2s
lote 34 (2000 series): 3.2s
lote 35 (2000 series): 3.2s
lo

## 4. Unir clusters a las features (solo tiene sentido correr esto con TEST_RAPIDO=False, todos los lotes procesados)

In [21]:
tb_clusters = pl.DataFrame({"fila_id": fila_ids, "cluster_id": asignaciones})
tb_clusters = tb_clusters.filter(pl.col("fila_id").is_not_nan())
tb_clusters = tb_clusters.with_columns(
    pl.col("fila_id").cast(pl.UInt32),
    pl.col("cluster_id").cast(pl.Int64)
)
tb_clusters = tb_clusters.filter(pl.col("cluster_id") >= 0)

df = df.join(tb_clusters, on="fila_id", how="inner")
print(df.shape)
print("clusters distintos:", df["cluster_id"].n_unique())

(16648065, 75)
clusters distintos: 1750


## 5. Un LightGBM por cluster (identico esquema a z615, adaptado a customer_id+product_id)
Con cientos de clusters esto puede tardar bastante. Se salta cualquier cluster con muy poca data.

In [22]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

CLAVE = ["customer_id", "product_id"]
H = PARAM['horizonte_meses']

df = df.sort(CLAVE + ["periodo"])
df = df.with_columns(
    pl.col("tn").shift(-H).over(CLAVE).alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m", "cluster_id", "fila_id"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["customer_id", "product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

params_fijos = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': PARAM['semilla']
}

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

In [23]:
predicciones_totales = []
clusters_saltados = 0

for cluster_id in sorted(df["cluster_id"].unique().to_list()):
    sub = df.filter(pl.col("cluster_id") == cluster_id)
    sub_valido = sub.filter(pl.col("tn_target").is_not_null())

    train = sub_valido.filter(pl.col("periodo_target_m") <= m_201910)
    valid = sub_valido.filter(
        (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
    )

    if train.height < PARAM['min_filas_por_cluster'] or valid.height < 5:
        clusters_saltados += 1
        continue

    train_pd = a_pandas(train)
    valid_pd = a_pandas(valid)

    X_train = train_pd[features]
    y_train = np.log1p(train_pd["tn_target"].clip(lower=0))
    X_valid = valid_pd[features]
    y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})

    modelo = lgb.train(
        params_fijos, dtrain, num_boost_round=1000,
        valid_sets=[dvalid], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    futuro = sub.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
    futuro_pd = futuro.select(features).to_pandas()
    for c in categoricas:
        futuro_pd[c] = futuro_pd[c].astype("category")

    pred_log = modelo.predict(futuro_pd, num_iteration=modelo.best_iteration)
    pred_tn = np.clip(np.expm1(pred_log), 0, None)

    res = futuro.select(["customer_id", "product_id"]).to_pandas()
    res["tn"] = pred_tn
    predicciones_totales.append(res)

print("clusters entrenados:", len(predicciones_totales), " clusters saltados (poca data):", clusters_saltados)

clusters entrenados: 1527  clusters saltados (poca data): 223


## 6. Combinar, SUMAR por product_id y armar submit

In [24]:
resultado_cp = pd.concat(predicciones_totales, ignore_index=True)
resultado = resultado_cp.groupby("product_id", as_index=False)["tn"].sum()

apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()
submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos (productos sin ninguna serie entrenada, revisar):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos (productos sin ninguna serie entrenada, revisar): 0
/home/ds/exp/LGB11_DTW_CP_TEST/LGB11_DTW_CP_TEST_submit.csv


,product_id,tn
0,20001,592.232474
1,20002,544.332570
2,20003,423.058080
3,20004,322.845464
4,20005,318.960161


In [25]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} DTW clusters cliente-producto")

100%|██████████| 18.6k/18.6k [00:00<00:00, 60.8kB/s]


97 submissions remaining today.
Successfully submitted to Labo III, 2026 BA